# Ders 09 - Metabiliş Tasarım Deseni


## Kurulum

Bu not defteri, Microsoft Agent Framework kullanılarak Metakognisyon tasarım desenini gösterir.

**Önkoşullar:**
- Azure OpenAI dağıtımının ortam değişkenleri aracılığıyla yapılandırılmış olması
- Azure CLI'nin kimlik doğrulamasının yapılmış olması (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Meta-biliş nedir?

Meta-biliş, **düşünme hakkında düşünmektir**. Yapay zeka ajanları bağlamında, bu şu özelliklere sahip ajanlar oluşturmak anlamına gelir:

- **Öz-yansıma** kendi çıktıları ve akıl yürütme süreçleri üzerinde
- **Hataları tespit etme** ve sessizce başarısız olmak yerine uygun şekilde toparlanma
- **Değerlendirme** yanıtlarının eksiksiz ve yardımcı olup olmadığını
- **Uyum sağlama** başlangıçtaki yaklaşım işe yaramadığında stratejilerini değiştirme (ör. bir yedek sisteme dönme)

Bir meta-bilişsel ajan sadece soruları cevaplamaz — kendi performansını izler ve anında ayarlamalar yapar.


## Birincil ve Yedek Araçlar

Yaygın bir metabiliş örüntüsü **geri dönüş stratejisi**dir. Ajan önce birincil aracı dener; eğer başarısız olursa (ör. 404 hatası), ajan hatayı fark eder ve şeffaf şekilde bir yedek araca geçer.

Bu, birincil hizmetlerin kullanılamayabileceği ve ajanların alternatif bir yol seçmeden önce sorunu kendi kendilerine teşhis etmeleri gerektiği gerçek dünya sistemlerini yansıtır.

Aşağıda iki uçuş arama aracını tanımlıyoruz:
- **Birincil** — Paris, Tokyo ve Barselona'yı kapsar
- **Yedek** — Berlin, Sidney ve New York'u kapsar


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Hata Kurtarmalı Öz-Değerlendirme Yapan Ajan

Aşağıdaki ajana öncelikle birincil uçuş sistemini denemesi, hataları tanıması ve şeffaf bir şekilde yedek sisteme geçmesi talimatı verilmiştir. Her yanıtın ardından, kullanıcının sorusunu tamamen yanıtlayıp yanıtlamadığını kısaca kendini değerlendirir.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Öz-Değerlendirme Deseni

Metakognisyonun başka bir yönü **öz-değerlendirme**dir: ayrı bir ajan (veya aynı ajanın ikinci bir geçişte yaptığı inceleme) bir yanıtı tamlık, doğruluk ve faydalılık açısından gözden geçirir.

Aşağıda üç boyutta seyahat acentesi yanıtlarını puanlayan bir `ResponseEvaluator` ajanı oluşturuyoruz.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Özet

Bu derste Microsoft Agent Framework kullanarak **metakognitif ajanlar** oluşturmayı öğrendiniz:

- **Öz-yansıtma**: Kendi akıl yürütmelerini izleyen ve ne olduğunu şeffaf şekilde ileten ajanlar.
- **Yedeklerle hata kurtarma**: Ajanın hataları (ör. 404 hataları) algıladığı ve otomatik olarak alternatif bir kaynağı denediği bir birincil + yedek araç deseni.
- **Öz-değerlendirme**: Yanıtları tamlık, doğruluk ve fayda açısından puanlayan ayrı bir değerlendirme ajanı.

Bu desenler ajanları daha sağlam, şeffaf ve güvenilir kılar — üretim dağıtımları için kritik nitelikler.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Feragatname:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluğu sağlamaya çalışsak da otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Orijinal belge, kendi dilindeki nüshası yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalar veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
